In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
def write_to_silver(df, target_table, merge_condition, columns_to_update):
    df = (
        df
            .withColumn("created_timestamp", F.current_timestamp())
            .withColumn("updated_timestamp", F.current_timestamp())
    )

    if not spark.catalog.tableExists(target_table):
        df.write.format("delta").mode("overwrite").saveAsTable(target_table)
    else:
        delta_table = DeltaTable.forName(spark, target_table)
        to_update = {column : f"s.{column}" for column in columns_to_update}
        to_update["updated_timestamp"] = "s.updated_timestamp"
        delta_table.alias("t").merge(
            df.alias("s"),
            merge_condition
        ).whenMatchedUpdate(
            condition="s.batch_id>=t.batch_id",
            set=to_update
        ).whenNotMatchedInsertAll().execute()